# Camada Gold — V-Commerce CRM 360

**Módulo 1 · Engenharia de Dados · Arquitetura Medalhão**

---

## Visão Geral

A camada Gold consolida e agrega os dados tratados da Silver para responder perguntas de negócio específicas. As tabelas produzidas aqui são **desnormalizadas por design** — o objetivo é que o backend FastAPI execute o mínimo de JOINs possível, já que cada tabela chega pronta para consumo pelos endpoints do CRM.

Os princípios aplicados em todas as tabelas Gold são:

- **Agregação orientada ao consumidor** — cada tabela responde a uma pergunta de negócio clara, mapeada a um stakeholder específico
- **Idempotência** — todas as escritas usam `mode=overwrite`, garantindo que reexecutar produza sempre o mesmo resultado
- **Rastreabilidade** — `timestamp_ingestion` registra o instante em que a tabela Gold foi gerada
- **Sem JOINs no backend** — métricas calculáveis a partir da Silver são materializadas aqui para evitar recomputação em tempo de consulta
- **Leitura exclusiva da Silver** — nenhuma tabela Gold lê de Bronze diretamente

---

## Tabelas produzidas neste notebook

| Tabela Gold | Granularidade | Stakeholder Principal | Descrição |
|---|---|---|---|
| `gold_cliente_360` | 1 linha por cliente | Fernanda Souza (Customer Success) | Visão 360 consolidada de cada cliente |
| `gold_kpis_vendas_mensal` | 1 linha por mês | Ricardo Alves (Diretor Comercial) | KPIs de vendas mensais para o dashboard |
| `gold_vendas_por_dimensao` | 1 linha por mês × região × categoria | Ricardo Alves (Diretor Comercial) | Drill-down de receita por dimensão |
| `gold_desempenho_produto` | 1 linha por produto | Marcelo Teixeira (Gerente de Produto) | Métricas individuais de cada produto |
| `gold_analise_suporte_por_tipo` | 1 linha por tipo de problema | Time de Suporte | Desempenho do SAC por categoria de problema |
| `gold_analise_suporte_por_agente` | 1 linha por agente | Gestão do SAC | Desempenho individual de cada agente |
| `gold_satisfacao_nps` | 1 linha por mês × categoria | Diretoria | NPS e satisfação consolidados por período e categoria |

---

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

catalogo      = 'vcommerce_catalog'
silver_schema = 'vcommerce_silver'
gold_schema   = 'vcommerce_gold'

spark.sql(f'USE CATALOG {catalogo}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {gold_schema}')
spark.sql(f'USE SCHEMA {gold_schema}')

print(f'Catálogo : {catalogo}')
print(f'Silver   : {silver_schema}')
print(f'Gold     : {gold_schema}')

---

## Tabela: `gold_cliente_360`

**Origem:** `silver.dim_clientes`, `silver.ft_pedidos`, `silver.ft_tickets_suporte`, `silver.ft_avaliacoes`  
**Destino:** `gold.gold_cliente_360`  
**Stakeholder:** Fernanda Souza — Diretora de Customer Success  
**Descrição:** Visão consolidada de cada cliente em uma única linha, agregando métricas de compra, suporte e satisfação. É a tabela que alimenta o perfil 360 do cliente no CRM.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `id_cliente` | string | dim_clientes | Chave primária |
| `nome_completo` | string | dim_clientes | Nome completo do cliente |
| `email` | string | dim_clientes | E-mail de contato |
| `regiao` | string | dim_clientes | Região geográfica |
| `origem` | string | dim_clientes | Canal de aquisição (Web, Indicação…) |
| `total_pedidos` | long | ft_pedidos | Total de pedidos realizados |
| `receita_total` | decimal | ft_pedidos | Soma de `valor_total` de todos os pedidos |
| `ticket_medio` | decimal | ft_pedidos | `receita_total / total_pedidos` |
| `data_primeiro_pedido` | date | ft_pedidos | Data do primeiro pedido |
| `data_ultimo_pedido` | date | ft_pedidos | Data do pedido mais recente |
| `metodo_pagamento_favorito` | string | ft_pedidos | Método de pagamento mais utilizado (moda) |
| `total_tickets` | long | ft_tickets_suporte | Total de tickets abertos |
| `taxa_resolucao` | decimal | ft_tickets_suporte | `tickets_resolvidos / total_tickets` |
| `nota_media_atendimento` | decimal | ft_tickets_suporte | Média de `nota_avaliacao` dos tickets |
| `nota_nps_media` | decimal | ft_avaliacoes | Média das notas NPS dadas pelo cliente |
| `categoria_nps_predominante` | string | ft_avaliacoes | Categoria NPS mais frequente (Promotor/Neutro/Detrator) |
| `nota_produto_media` | decimal | ft_avaliacoes | Média das notas de produto dadas pelo cliente |
| `segmento_cliente` | string | derivado | VIP / Ativo / Em risco / Inativo |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

### Regras de segmentação (`segmento_cliente`)

| Segmento | Critério |
|---|---|
| `VIP` | `receita_total >= 2000` ou `total_pedidos >= 10` |
| `Ativo` | Último pedido há no máximo 90 dias (e não VIP) |
| `Em risco` | Último pedido entre 90 e 180 dias atrás |
| `Inativo` | Último pedido há mais de 180 dias ou sem pedidos |

In [0]:
# ─── Leitura das tabelas Silver ───────────────────────────────────────────────
clientes  = spark.table(f'{silver_schema}.dim_clientes')
pedidos   = spark.table(f'{silver_schema}.ft_pedidos')
tickets   = spark.table(f'{silver_schema}.ft_tickets_suporte')
avaliacoes = spark.table(f'{silver_schema}.ft_avaliacoes')

# ─── Agregação de pedidos por cliente ─────────────────────────────────────────
# metodo_pagamento_favorito: modo (valor mais frequente) por cliente
w_pagto = Window.partitionBy('id_cliente').orderBy(F.desc('freq_pagto'))

metodo_fav = (
    pedidos
    .groupBy('id_cliente', 'metodo_pagamento')
    .agg(F.count('*').alias('freq_pagto'))
    .withColumn('rank_pagto', F.row_number().over(w_pagto))
    .filter(F.col('rank_pagto') == 1)
    .select('id_cliente', F.col('metodo_pagamento').alias('metodo_pagamento_favorito'))
)

agg_pedidos = (
    pedidos
    .groupBy('id_cliente')
    .agg(
        F.count('id_pedido').alias('total_pedidos'),
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
        F.min(F.to_date('data_pedido')).alias('data_primeiro_pedido'),
        F.max(F.to_date('data_pedido')).alias('data_ultimo_pedido'),
    )
    .join(metodo_fav, on='id_cliente', how='left')
)

# ─── Agregação de tickets por cliente ─────────────────────────────────────────
agg_tickets = (
    tickets
    .groupBy('id_cliente')
    .agg(
        F.count('ticket_id').alias('total_tickets'),
        F.round(
            F.sum(F.when(F.col('resolvido') == True, 1).otherwise(0)) / F.count('ticket_id'),
            4
        ).alias('taxa_resolucao'),
        F.round(F.avg('nota_avaliacao'), 2).alias('nota_media_atendimento'),
    )
)

# ─── Agregação de avaliações por cliente ──────────────────────────────────────
# categoria_nps_predominante: modo por cliente
w_nps = Window.partitionBy('id_cliente').orderBy(F.desc('freq_nps'))

nps_fav = (
    avaliacoes
    .groupBy('id_cliente', 'categoria_nps')
    .agg(F.count('*').alias('freq_nps'))
    .withColumn('rank_nps', F.row_number().over(w_nps))
    .filter(F.col('rank_nps') == 1)
    .select('id_cliente', F.col('categoria_nps').alias('categoria_nps_predominante'))
)

agg_avaliacoes = (
    avaliacoes
    .groupBy('id_cliente')
    .agg(
        F.round(F.avg('nota_nps'), 2).alias('nota_nps_media'),
        F.round(F.avg('nota_produto'), 2).alias('nota_produto_media'),
    )
    .join(nps_fav, on='id_cliente', how='left')
)

# ─── Base de clientes + joins das agregações ──────────────────────────────────
base = clientes.select(
    'id_cliente', 'nome_completo', 'email', 'regiao', 'origem'
)

gold_c360 = (
    base
    .join(agg_pedidos,   on='id_cliente', how='left')
    .join(agg_tickets,   on='id_cliente', how='left')
    .join(agg_avaliacoes, on='id_cliente', how='left')
)

# ─── Derivação: segmento_cliente ──────────────────────────────────────────────
dias_inativo = F.datediff(F.current_date(), F.col('data_ultimo_pedido'))

gold_c360 = gold_c360.withColumn(
    'segmento_cliente',
    F.when(
        (F.col('receita_total') >= 2000) | (F.col('total_pedidos') >= 10), 'VIP'
    ).when(
        dias_inativo <= 90, 'Ativo'
    ).when(
        (dias_inativo > 90) & (dias_inativo <= 180), 'Em risco'
    ).otherwise('Inativo')
)

# ─── Timestamp de geração ─────────────────────────────────────────────────────
gold_c360 = gold_c360.withColumn('timestamp_ingestion', F.current_timestamp())

# ─── Escrita ──────────────────────────────────────────────────────────────────
gold_c360.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_cliente_360')

total = gold_c360.count()
print(f'gold_cliente_360 gravada: {total:,} clientes')
print('Distribuição por segmento:')
gold_c360.groupBy('segmento_cliente').count().orderBy(F.desc('count')).show()

---

## Tabela: `gold_kpis_vendas_mensal`

**Origem:** `silver.ft_pedidos`, `silver.dim_clientes`  
**Destino:** `gold.gold_kpis_vendas_mensal`  
**Stakeholder:** Ricardo Alves — Diretor Comercial  
**Descrição:** Uma linha por mês com os principais KPIs de vendas. Alimenta o gráfico de série temporal e o painel de KPIs do dashboard.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `ano_mes` | string | ft_pedidos | Período no formato `YYYY-MM` |
| `receita_total` | decimal | ft_pedidos | Soma de `valor_total` dos pedidos não cancelados |
| `total_pedidos` | long | ft_pedidos | Total de pedidos no período |
| `ticket_medio` | decimal | ft_pedidos | `receita_total / total_pedidos` |
| `total_clientes_ativos` | long | ft_pedidos | Clientes distintos com pedido no período |
| `novos_clientes` | long | ft_pedidos | Clientes cujo primeiro pedido ocorreu neste mês |
| `pedidos_cancelados` | long | ft_pedidos | Pedidos com status cancelado |
| `taxa_cancelamento` | decimal | ft_pedidos | `pedidos_cancelados / total_pedidos` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
pedidos = spark.table(f'{silver_schema}.ft_pedidos')

# ─── Primeiro pedido por cliente (para calcular novos_clientes) ───────────────
primeiro_pedido = (
    pedidos
    .groupBy('id_cliente')
    .agg(F.min('ano_mes').alias('ano_mes_primeiro_pedido'))
)

# ─── Novos clientes por mês ───────────────────────────────────────────────────
novos_por_mes = (
    primeiro_pedido
    .groupBy('ano_mes_primeiro_pedido')
    .agg(F.count('id_cliente').alias('novos_clientes'))
    .withColumnRenamed('ano_mes_primeiro_pedido', 'ano_mes')
)

# ─── KPIs mensais ─────────────────────────────────────────────────────────────
pedidos_nao_cancelados = pedidos.filter(F.col('status') != 'Cancelado')

kpis = (
    pedidos
    .groupBy('ano_mes')
    .agg(
        F.round(F.sum(
            F.when(F.col('status') != 'Cancelado', F.col('valor_total')).otherwise(0)
        ), 2).alias('receita_total'),
        F.count('id_pedido').alias('total_pedidos'),
        F.countDistinct('id_cliente').alias('total_clientes_ativos'),
        F.sum(
            F.when(F.col('status') == 'Cancelado', 1).otherwise(0)
        ).alias('pedidos_cancelados'),
    )
    .join(novos_por_mes, on='ano_mes', how='left')
    .withColumn('novos_clientes', F.coalesce(F.col('novos_clientes'), F.lit(0)))
    .withColumn(
        'ticket_medio',
        F.round(F.col('receita_total') / F.col('total_pedidos'), 2)
    )
    .withColumn(
        'taxa_cancelamento',
        F.round(F.col('pedidos_cancelados') / F.col('total_pedidos'), 4)
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .orderBy('ano_mes')
)

kpis.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_kpis_vendas_mensal')

print(f'gold_kpis_vendas_mensal gravada: {kpis.count()} meses')
kpis.select('ano_mes', 'receita_total', 'total_pedidos', 'ticket_medio', 'novos_clientes', 'taxa_cancelamento').show(5)

---

## Tabela: `gold_vendas_por_dimensao`

**Origem:** `silver.ft_pedidos`, `silver.dim_produtos`, `silver.dim_clientes`  
**Destino:** `gold.gold_vendas_por_dimensao`  
**Stakeholder:** Ricardo Alves — Diretor Comercial  
**Descrição:** Drill-down de vendas com granularidade `ano_mes × regiao × categoria`. Permite ao Diretor Comercial investigar qual região ou categoria está impactando os resultados de um determinado período.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `ano_mes` | string | ft_pedidos | Período no formato `YYYY-MM` |
| `regiao` | string | dim_clientes | Região geográfica do cliente |
| `categoria` | string | dim_produtos | Categoria do produto vendido |
| `receita_total` | decimal | ft_pedidos | Soma de `valor_total` (pedidos não cancelados) |
| `total_pedidos` | long | ft_pedidos | Total de pedidos na combinação |
| `ticket_medio` | decimal | derivado | `receita_total / total_pedidos` |
| `quantidade_itens_vendidos` | long | ft_pedidos | Soma de `quantidade` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
pedidos   = spark.table(f'{silver_schema}.ft_pedidos')
produtos  = spark.table(f'{silver_schema}.dim_produtos').select('id_produto', 'categoria')
clientes  = spark.table(f'{silver_schema}.dim_clientes').select('id_cliente', 'regiao')

dim = (
    pedidos
    .filter(F.col('status') != 'Cancelado')
    .join(produtos, on='id_produto', how='left')
    .join(clientes, on='id_cliente', how='left')
    .groupBy('ano_mes', 'regiao', 'categoria')
    .agg(
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.count('id_pedido').alias('total_pedidos'),
        F.sum('quantidade').alias('quantidade_itens_vendidos'),
    )
    .withColumn(
        'ticket_medio',
        F.round(F.col('receita_total') / F.col('total_pedidos'), 2)
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .orderBy('ano_mes', 'regiao', 'categoria')
)

dim.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_vendas_por_dimensao')

print(f'gold_vendas_por_dimensao gravada: {dim.count():,} combinações')
dim.orderBy(F.desc('receita_total')).show(5)

---

## Tabela: `gold_desempenho_produto`

**Origem:** `silver.dim_produtos`, `silver.ft_pedidos`, `silver.ft_avaliacoes`, `silver.ft_tickets_suporte`  
**Destino:** `gold.gold_desempenho_produto`  
**Stakeholder:** Marcelo Teixeira — Gerente de Produto  
**Descrição:** Uma linha por produto com métricas consolidadas de vendas, satisfação e suporte. O campo `ratio_ticket_por_venda` responde à dor central do Gerente de Produto: identificar produtos que vendem bem mas geram custo operacional desproporcional via tickets de suporte.

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `id_produto` | string | dim_produtos | Chave primária |
| `nome_produto` | string | dim_produtos | Nome do produto |
| `categoria` | string | dim_produtos | Categoria normalizada |
| `preco` | decimal | dim_produtos | Preço de tabela atual |
| `fornecedor` | string | dim_produtos | Fornecedor do produto |
| `estoque_disponivel` | int | dim_produtos | Unidades em estoque |
| `ativo` | boolean | dim_produtos | Se o produto está ativo no catálogo |
| `receita_total` | decimal | ft_pedidos | Receita gerada pelo produto |
| `qtd_vendida` | long | ft_pedidos | Total de unidades vendidas |
| `ticket_medio` | decimal | ft_pedidos | Receita média por pedido |
| `nota_media_avaliacao` | decimal | ft_avaliacoes | Média das notas de produto |
| `qtd_avaliacoes` | long | ft_avaliacoes | Total de avaliações recebidas |
| `nota_nps_media` | decimal | ft_avaliacoes | Média das notas NPS vinculadas ao produto |
| `qtd_tickets_gerados` | long | ft_tickets_suporte | Tickets de suporte originados de pedidos deste produto |
| `tipo_problema_mais_frequente` | string | ft_tickets_suporte | Problema mais comum relatado para este produto |
| `ratio_ticket_por_venda` | decimal | derivado | `qtd_tickets_gerados / qtd_vendida` — índice de problema por unidade vendida |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
produtos   = spark.table(f'{silver_schema}.dim_produtos')
pedidos    = spark.table(f'{silver_schema}.ft_pedidos')
avaliacoes = spark.table(f'{silver_schema}.ft_avaliacoes')
tickets    = spark.table(f'{silver_schema}.ft_tickets_suporte')

# ─── Métricas de vendas por produto ───────────────────────────────────────────
agg_vendas = (
    pedidos
    .filter(F.col('status') != 'Cancelado')
    .groupBy('id_produto')
    .agg(
        F.round(F.sum('valor_total'), 2).alias('receita_total'),
        F.sum('quantidade').alias('qtd_vendida'),
        F.round(F.avg('valor_total'), 2).alias('ticket_medio'),
    )
)

# ─── Métricas de avaliações por produto ───────────────────────────────────────
agg_aval = (
    avaliacoes
    .groupBy('id_produto')
    .agg(
        F.round(F.avg('nota_produto'), 2).alias('nota_media_avaliacao'),
        F.count('id_avaliacao').alias('qtd_avaliacoes'),
        F.round(F.avg('nota_nps'), 2).alias('nota_nps_media'),
    )
)

# ─── Tickets via pedidos → produto ────────────────────────────────────────────
# Tickets não têm id_produto diretamente: linkamos via id_pedido → ft_pedidos
tickets_com_produto = (
    tickets
    .join(
        pedidos.select('id_pedido', 'id_produto'),
        on='id_pedido', how='left'
    )
)

# Tipo de problema mais frequente por produto
w_prob = Window.partitionBy('id_produto').orderBy(F.desc('freq_prob'))

tipo_fav = (
    tickets_com_produto
    .groupBy('id_produto', 'tipo_problema')
    .agg(F.count('*').alias('freq_prob'))
    .withColumn('rank_prob', F.row_number().over(w_prob))
    .filter(F.col('rank_prob') == 1)
    .select('id_produto', F.col('tipo_problema').alias('tipo_problema_mais_frequente'))
)

agg_tickets = (
    tickets_com_produto
    .groupBy('id_produto')
    .agg(F.count('ticket_id').alias('qtd_tickets_gerados'))
    .join(tipo_fav, on='id_produto', how='left')
)

# ─── Join final ───────────────────────────────────────────────────────────────
gold_prod = (
    produtos.select(
        'id_produto', 'nome_produto', 'categoria', 'preco',
        'fornecedor', 'estoque_disponivel', 'ativo'
    )
    .join(agg_vendas,   on='id_produto', how='left')
    .join(agg_aval,     on='id_produto', how='left')
    .join(agg_tickets,  on='id_produto', how='left')
    .withColumn(
        'ratio_ticket_por_venda',
        F.round(
            F.col('qtd_tickets_gerados') / F.col('qtd_vendida'),
            4
        )
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
)

gold_prod.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_desempenho_produto')

print(f'gold_desempenho_produto gravada: {gold_prod.count():,} produtos')
print('Top 5 por ratio_ticket_por_venda:')
gold_prod.orderBy(F.desc('ratio_ticket_por_venda')).select(
    'nome_produto', 'qtd_vendida', 'qtd_tickets_gerados', 'ratio_ticket_por_venda', 'tipo_problema_mais_frequente'
).show(5, truncate=False)

---

## Tabelas: `gold_analise_suporte_por_tipo` e `gold_analise_suporte_por_agente`

**Origem:** `silver.ft_tickets_suporte`, `silver.dim_tipos_problema`, `silver.dim_agentes_suporte`  
**Destino:** `gold.gold_analise_suporte_por_tipo`, `gold.gold_analise_suporte_por_agente`  
**Stakeholder:** Time de Suporte / Gestão do SAC  
**Descrição:** Duas tabelas complementares que alimentam o dashboard de suporte. A primeira oferece visão por tipo de problema; a segunda, visão por agente.

### Schema — `gold_analise_suporte_por_tipo`

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `tipo_problema` | string | ft_tickets_suporte | Tipo de problema canônico |
| `categoria_problema` | string | dim_tipos_problema | Categoria de negócio do problema |
| `total_tickets` | long | ft_tickets_suporte | Total de tickets deste tipo |
| `tickets_resolvidos` | long | ft_tickets_suporte | Tickets com resolução registrada |
| `taxa_resolucao` | decimal | derivado | `tickets_resolvidos / total_tickets` |
| `tempo_medio_resolucao_horas` | decimal | ft_tickets_suporte | Média de `tempo_resolucao_horas` (apenas resolvidos) |
| `nota_media_atendimento` | decimal | ft_tickets_suporte | Média de `nota_avaliacao` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

### Schema — `gold_analise_suporte_por_agente`

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `agente_suporte` | string | ft_tickets_suporte | Nome do agente |
| `total_tickets` | long | ft_tickets_suporte | Total de tickets atendidos |
| `tickets_resolvidos` | long | ft_tickets_suporte | Tickets com resolução registrada |
| `taxa_resolucao` | decimal | derivado | `tickets_resolvidos / total_tickets` |
| `tempo_medio_resolucao_horas` | decimal | ft_tickets_suporte | Média de `tempo_resolucao_horas` |
| `nota_media_atendimento` | decimal | ft_tickets_suporte | Média de `nota_avaliacao` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
tickets      = spark.table(f'{silver_schema}.ft_tickets_suporte')
dim_prob     = spark.table(f'{silver_schema}.dim_tipos_problema')

suporte_tipo = (
    tickets
    .groupBy('tipo_problema')
    .agg(
        F.count('ticket_id').alias('total_tickets'),
        F.sum(F.when(F.col('resolvido') == True, 1).otherwise(0)).alias('tickets_resolvidos'),
        F.round(F.avg(
            F.when(F.col('resolvido') == True, F.col('tempo_resolucao_horas'))
        ), 2).alias('tempo_medio_resolucao_horas'),
        F.round(F.avg('nota_avaliacao'), 2).alias('nota_media_atendimento'),
    )
    .join(dim_prob, on='tipo_problema', how='left')
    .withColumn(
        'taxa_resolucao',
        F.round(F.col('tickets_resolvidos') / F.col('total_tickets'), 4)
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .select(
        'tipo_problema', 'categoria_problema', 'total_tickets',
        'tickets_resolvidos', 'taxa_resolucao',
        'tempo_medio_resolucao_horas', 'nota_media_atendimento',
        'timestamp_ingestion'
    )
    .orderBy(F.desc('total_tickets'))
)

suporte_tipo.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_analise_suporte_por_tipo')

print(f'gold_analise_suporte_por_tipo gravada: {suporte_tipo.count()} tipos de problema')
suporte_tipo.show(truncate=False)

In [0]:
tickets = spark.table(f'{silver_schema}.ft_tickets_suporte')

suporte_agente = (
    tickets
    .groupBy('agente_suporte')
    .agg(
        F.count('ticket_id').alias('total_tickets'),
        F.sum(F.when(F.col('resolvido') == True, 1).otherwise(0)).alias('tickets_resolvidos'),
        F.round(F.avg(
            F.when(F.col('resolvido') == True, F.col('tempo_resolucao_horas'))
        ), 2).alias('tempo_medio_resolucao_horas'),
        F.round(F.avg('nota_avaliacao'), 2).alias('nota_media_atendimento'),
    )
    .withColumn(
        'taxa_resolucao',
        F.round(F.col('tickets_resolvidos') / F.col('total_tickets'), 4)
    )
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .select(
        'agente_suporte', 'total_tickets', 'tickets_resolvidos',
        'taxa_resolucao', 'tempo_medio_resolucao_horas',
        'nota_media_atendimento', 'timestamp_ingestion'
    )
    .orderBy(F.desc('total_tickets'))
)

suporte_agente.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_analise_suporte_por_agente')

print(f'gold_analise_suporte_por_agente gravada: {suporte_agente.count()} agentes')
suporte_agente.show(10, truncate=False)

---

## Tabela: `gold_satisfacao_nps`

**Origem:** `silver.ft_avaliacoes`, `silver.ft_pedidos`, `silver.dim_produtos`  
**Destino:** `gold.gold_satisfacao_nps`  
**Stakeholder:** Diretoria / Qualquer área com visão executiva  
**Descrição:** Uma linha por `ano_mes × categoria` com métricas de NPS e satisfação consolidadas. O `nps_score` é calculado como `% Promotores − % Detratores`, na escala de −100 a +100.

### Classificação NPS (regra aplicada na Silver)

| Categoria | Nota NPS |
|---|---|
| `Promotor` | 9 ou 10 |
| `Neutro` | 7 ou 8 |
| `Detrator` | 0 a 6 |

### Schema

| Coluna | Tipo | Origem | Descrição |
|---|---|---|---|
| `ano_mes` | string | ft_pedidos | Período no formato `YYYY-MM` |
| `categoria` | string | dim_produtos | Categoria do produto avaliado |
| `total_avaliacoes` | long | ft_avaliacoes | Total de avaliações no período e categoria |
| `nota_produto_media` | decimal | ft_avaliacoes | Média das notas do produto (0–5) |
| `nota_nps_media` | decimal | ft_avaliacoes | Média das notas NPS (0–10) |
| `qtd_promotores` | long | ft_avaliacoes | Avaliações com `categoria_nps = 'Promotor'` |
| `qtd_neutros` | long | ft_avaliacoes | Avaliações com `categoria_nps = 'Neutro'` |
| `qtd_detratores` | long | ft_avaliacoes | Avaliações com `categoria_nps = 'Detrator'` |
| `pct_promotores` | decimal | derivado | `qtd_promotores / total_avaliacoes × 100` |
| `pct_neutros` | decimal | derivado | `qtd_neutros / total_avaliacoes × 100` |
| `pct_detratores` | decimal | derivado | `qtd_detratores / total_avaliacoes × 100` |
| `nps_score` | decimal | derivado | `pct_promotores − pct_detratores` (−100 a +100) |
| `pct_recomenda` | decimal | derivado | `% de avaliações com recomenda = true` |
| `timestamp_ingestion` | timestamp | derivado | Instante de geração desta tabela Gold |

In [0]:
avaliacoes = spark.table(f'{silver_schema}.ft_avaliacoes')
pedidos    = spark.table(f'{silver_schema}.ft_pedidos').select('id_pedido', 'ano_mes')
produtos   = spark.table(f'{silver_schema}.dim_produtos').select('id_produto', 'categoria')

# ─── Enriquecer avaliações com ano_mes (via pedido) e categoria (via produto) ─
aval_enriquecida = (
    avaliacoes
    .join(pedidos,  on='id_pedido',  how='left')
    .join(produtos, on='id_produto', how='left')
)

# ─── Agregação por ano_mes × categoria ────────────────────────────────────────
nps = (
    aval_enriquecida
    .groupBy('ano_mes', 'categoria')
    .agg(
        F.count('id_avaliacao').alias('total_avaliacoes'),
        F.round(F.avg('nota_produto'), 2).alias('nota_produto_media'),
        F.round(F.avg('nota_nps'), 2).alias('nota_nps_media'),
        F.sum(F.when(F.col('categoria_nps') == 'Promotor', 1).otherwise(0)).alias('qtd_promotores'),
        F.sum(F.when(F.col('categoria_nps') == 'Neutro',   1).otherwise(0)).alias('qtd_neutros'),
        F.sum(F.when(F.col('categoria_nps') == 'Detrator', 1).otherwise(0)).alias('qtd_detratores'),
        F.sum(F.when(F.col('recomenda') == True, 1).otherwise(0)).alias('qtd_recomenda'),
    )
    # ─── Percentuais e NPS Score ──────────────────────────────────────────────
    .withColumn('pct_promotores', F.round(F.col('qtd_promotores') / F.col('total_avaliacoes') * 100, 2))
    .withColumn('pct_neutros',    F.round(F.col('qtd_neutros')    / F.col('total_avaliacoes') * 100, 2))
    .withColumn('pct_detratores', F.round(F.col('qtd_detratores') / F.col('total_avaliacoes') * 100, 2))
    .withColumn('nps_score',      F.round(F.col('pct_promotores') - F.col('pct_detratores'), 2))
    .withColumn('pct_recomenda',  F.round(F.col('qtd_recomenda')  / F.col('total_avaliacoes') * 100, 2))
    .withColumn('timestamp_ingestion', F.current_timestamp())
    .drop('qtd_recomenda')
    .orderBy('ano_mes', 'categoria')
)

nps.write.format('delta').mode('overwrite').saveAsTable(f'{gold_schema}.gold_satisfacao_nps')

print(f'gold_satisfacao_nps gravada: {nps.count():,} combinações ano_mes × categoria')
print('NPS Score por categoria (média geral):')
nps.groupBy('categoria').agg(
    F.round(F.avg('nps_score'), 1).alias('nps_medio'),
    F.sum('total_avaliacoes').alias('total_avaliacoes')
).orderBy(F.desc('nps_medio')).show(truncate=False)

---

## Exportação para CSV

As tabelas Gold são exportadas em formato CSV para popular o banco SQLite local que serve o backend FastAPI do CRM.  
Ajuste o caminho `export_path` conforme o destino configurado no ambiente Databricks (DBFS ou Volume).

In [0]:
export_path = '/Volumes/vcommerce_catalog/vcommerce_gold/gold_exports'

GOLD_TABLES = [
    'gold_cliente_360',
    'gold_kpis_vendas_mensal',
    'gold_vendas_por_dimensao',
    'gold_desempenho_produto',
    'gold_analise_suporte_por_tipo',
    'gold_analise_suporte_por_agente',
    'gold_satisfacao_nps',
]

for table in GOLD_TABLES:
    (
        spark.table(f'{gold_schema}.{table}')
        .coalesce(1)   # garante um único arquivo CSV por tabela
        .write
        .mode('overwrite')
        .option('header', 'true')
        .csv(f'{export_path}/{table}')
    )
    print(f'  Exportada: {table}')

print(f'\nExportação concluída em: {export_path}')